In [7]:
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_nomic.embeddings import NomicEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
local_llm = "llava"
llm = ChatOllama(model=local_llm, temperature=0)

In [2]:
loader = PyPDFLoader("test-documents/future_of_work.pdf")
pages = loader.load_and_split()
doc_splits = text_splitter.split_documents(pages)

vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=NomicEmbeddings(model="nomic-embed-text-v1.5", inference_mode="local"),
)
retriever = vectorstore.as_retriever()

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 97 0 (offset 0)
Failed to load libllamamodel-mainline-cuda.so: dlopen: libcudart.so.11.0: cannot open shared object file: No such file or directory
Failed to load libllamamodel-mainline-cuda-avxonly.so: dlopen: libcudart.so.11.0: cannot open shared object file: No such file or directory


In [3]:
### Retrieval Grader

from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# LLM
llm = ChatOllama(model=local_llm, format="json", temperature=0)

prompt = PromptTemplate(
    template="""<|begin_of_text|><|start_header_id|>system<|end_header_id|> You are a grader assessing relevance 
    of a retrieved document to a user question. If the document contains keywords related to the user question, 
    grade it as relevant. It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question. \n
    Provide the binary score as a JSON with a single key 'score' and no premable or explanation.
     <|eot_id|><|start_header_id|>user<|end_header_id|>
    Here is the retrieved document: \n\n {document} \n\n
    Here is the user question: {question} \n <|eot_id|><|start_header_id|>assistant<|end_header_id|>
    """,
    input_variables=["question", "document"],
)

retrieval_grader = prompt | llm | JsonOutputParser()
question = "the future of work"
docs = retriever.invoke(question)
doc_txt = docs[1].page_content
print(retrieval_grader.invoke({"question": question, "document": doc_txt}))

{'score': 'yes'}


In [ ]:
### Generate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# Prompt
prompt = PromptTemplate(
    template="""
    <|begin_of_text|>
    
    <|start_header_id|>system<|end_header_id|>
    You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the question. 
    Reply with the exact wording retrieved from the context.
    If you don't know the answer, just say that you don't know. 
    You'll be provided the maximum number of sentence to keep the answer concise
    <|eot_id|>
    
    <|start_header_id|>user<|end_header_id|>
    Question: {question} 
    Context: {context}
    Answer Length: {length}
    Answer: <|eot_id|><|start_header_id|>assistant<|end_header_id|>
    """,
    input_variables=["question", "document"],
)

llm = ChatOllama(model=local_llm, temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
question = "What would addressing social and economic inequality be crucial to?"
docs = retriever.invoke(question)
generation = rag_chain.invoke({"context": docs, "question": question, "length":1})
print(generation)

 Addressing social and economic inequality will be crucial to sustainable, inclusive growth. 


In [27]:
docs

[Document(metadata={'page': 10, 'source': 'test-documents/future_of_work.pdf'}, page_content='inclusion. One statistic that brings the message home: up to 40percent of GDP growth in the US economy between 1960 and2010 can be attributed to an uptick in the participation of womenand people of color in the labor force through improved talentallocation.While studies show that companies that make more e"orts atdiversity, equity, and inclusion perform better, challengesremain. Job losses during the pandemic disproportionatelya"ected diverse populations, and some women opted out of theworkforce\xa0given school closures, a lack of childcare options, orother factors.Di"erent populations will have di"erent needs, andunderstanding the issues for Black Americans, Latinos inAmerica, Asian Americans, and LGBTQ+\xa0andtransgender\xa0employees (to take just a few examples) can help incrafting plans to make organizations more equitable andinclusive. The concept of intersectionality\xa0is also crucial: 

In [2]:
from langchain_community.document_loaders.image import UnstructuredImageLoader

loader = UnstructuredImageLoader("test-documents/miku.png")

data = loader.load()

data[0]

/root/miniconda3/envs/main/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Document(metadata={'source': 'test-documents/miku.png'}, page_content='')

In [21]:
import base64
from mimetypes import guess_type
from langchain_core.prompts.chat import HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
import json

# Function to encode a local image into data URL 
def local_image_to_data_url(image_path):
    mime_type, _ = guess_type(image_path)
    # Default to png
    if mime_type is None:
        mime_type = 'image/png'

    # Read and encode the image file
    with open(image_path, "rb") as image_file:
        base64_encoded_data = base64.b64encode(image_file.read()).decode('utf-8')

    # Construct the data URL
    return f"data:{mime_type};base64,{base64_encoded_data}"

answer_format = '''
{
  "summary": "A brief overall description of the scene and primary subjects.",
  "mood": "The general emotional atmosphere (e.g., joyful, tense, peaceful).",
  "tone": "The tone conveyed by the colors, lighting, and expressions (e.g., warm, cold, vibrant).",
  "subjects": [
    {
      "subject_description": "A short description of a primary person or object in the scene.",
      "clothing": "Detailed description of clothing style, colors, and any unique accessories.",
      "facial_expression": "Description of the subject's facial expression and any emotions conveyed.",
      "body_language": "Description of body posture or gestures that indicate mood or intention.",
      "color_palette": "Main colors associated with this subject, both in clothing and nearby elements."
    }
  ],
  "setting": "Description of the environment, background elements, and any notable objects.",
  "lighting": "Characteristics of the lighting, such as brightness, contrast, and any directional emphasis.",
  "color_composition": "Analysis of the overall color scheme and how it affects the mood."
}

'''

prompt_template =  HumanMessagePromptTemplate.from_template(
            template=[
                {
                    "type": "text", 
                    "text": '''
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are an assistant designed to observe and analyze surroundings and human interactions in visual scenes. 
Use the following guidance to provide a detailed breakdown of the image's elements. 
Focus on specifics like clothing, facial expressions, mood, tone, and color composition. 
Your output should be in the format of a JSON with the keys specified below. 
Only provide information you can directly observe in the image, and leave unknown details blank. 
If something is ambiguous, mark it as "uncertain."
Output the analysis in the following JSON format:
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Answer format: {answer_format}
Answer: <|eot_id|><|start_header_id|>assistant<|end_header_id|>
'''},
                {
                    "type": "image_url",
                    "image_url": "{encoded_image_url}",
                },
            ]
        )

summarize_image_prompt = ChatPromptTemplate.from_messages([prompt_template])

llm = ChatOllama(model="llava-llama3", temperature=0, format='json')
image_chain = summarize_image_prompt | llm | JsonOutputParser()

img_file = "test-documents/kk.jpg"
page3_encoded = local_image_to_data_url(img_file)

response = image_chain.invoke(input={"encoded_image_url":page3_encoded, "answer_format":answer_format})
with open("kk.json", 'w') as f:
  json.dump(response, f)

In [ ]:
### Generate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# Prompt
prompt = PromptTemplate(
    template="""
    <|begin_of_text|>
    
    <|start_header_id|>system<|end_header_id|>
    You are an assistant for an image captioning task
    You'll be given an image as a context a question related to it
    Answer the question based on the information provided in the image
    <|eot_id|>
    
    <|start_header_id|>user<|end_header_id|>
    Question: {question} 
    Context: {context}
    Answer Length: {length}
    Answer: <|eot_id|><|start_header_id|>assistant<|end_header_id|>
    """,
    input_variables=["question", "document"],
)

llm = ChatOllama(model=local_llm, temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
question = "What would addressing social and economic inequality be crucial to?"
docs = retriever.invoke(question)
generation = rag_chain.invoke({"context": docs, "question": question, "length":1})
print(generation)

In [ ]:
"<|im_start|>user <image>\n<prompt1><|im_end|><|im_start|>assistant <answer1><|im_end|><|im_start|>user <image>\n<prompt1><|im_end|><|im_start|>assistant "

In [18]:
import base64
from mimetypes import guess_type
from langchain_core.prompts.chat import HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
import json

# Function to encode a local image into data URL 
def local_image_to_data_url(image_path):
    mime_type, _ = guess_type(image_path)
    # Default to png
    if mime_type is None:
        mime_type = 'image/png'

    # Read and encode the image file
    with open(image_path, "rb") as image_file:
        base64_encoded_data = base64.b64encode(image_file.read()).decode('utf-8')

    # Construct the data URL
    return f"data:{mime_type};base64,{base64_encoded_data}"

answer_format = '''
{
  "summary": "A brief overall description of the scene and primary subjects.",
  "mood": "The general emotional atmosphere (e.g., joyful, tense, peaceful).",
  "tone": "The tone conveyed by the colors, lighting, and expressions (e.g., warm, cold, vibrant).",
  "subjects": [
    {
      "subject_description": "A short description of a primary person or object in the scene.",
      "clothing": "Detailed description of clothing style, colors, and any unique accessories.",
      "facial_expression": "Description of the subject's facial expression and any emotions conveyed.",
      "body_language": "Description of body posture or gestures that indicate mood or intention.",
      "color_palette": "Main colors associated with this subject, both in clothing and nearby elements."
    }
  ],
  "setting": "Description of the environment, background elements, and any notable objects.",
  "lighting": "Characteristics of the lighting, such as brightness, contrast, and any directional emphasis.",
  "color_composition": "Analysis of the overall color scheme and how it affects the mood."
}

'''

prompt_template =  HumanMessagePromptTemplate.from_template(
            template=[
                {
                    "type": "text", 
                    "text": '''
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
What's in this image? {encoded_image_url}
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Answer: <|eot_id|><|start_header_id|>assistant<|end_header_id|>
'''},
            ]
        )

summarize_image_prompt = ChatPromptTemplate.from_messages([prompt_template])

llm = ChatOllama(model="llava", temperature=0)
image_chain = summarize_image_prompt | llm | StrOutputParser()

img_file = "test-documents/kanata.jpg"
encoded = local_image_to_data_url(img_file)

response = image_chain.invoke(input={"encoded_image_url":encoded})

FileNotFoundError: [Errno 2] No such file or directory: 'images/miku.png'